In [15]:
import numpy as np
import matplotlib.pyplot as plt
import astropy
from astropy import units as u
#from astropy.coordinates import rotation_matrix
import astropy.modeling.rotations as rot
import ugradio

In [1]:
from coords import gal

# gal.to_AA(long=120,lat=0)

In [4]:
import numpy as np
from astropy.coordinates import SkyCoord, EarthLocation, AltAz
import astropy.units as u
import time

def to_AA(long, lat):
        """
        This function takes in galatic coordinates in latitude and longitude and uses ASTROPY to return a tuple in Azimuth and Altitude based on the latitude at NCH.
        """

        # Galatic Coordinates
        l = long * u.deg
        b = lat * u.deg

        # Convert to ICRS (Ra/Dec)
        coord_gal = SkyCoord(l=l, b=b, frame='galactic')
        coord_icrs = coord_gal.icrs

        # Observer Location (Berkeley)
        location = EarthLocation(lat=37.87*u.deg, lon=-122.27*u.deg, height=0*u.m)

        # Time
        t = time.time()

        # Convert to AltAz
        altaz = coord_icrs.transform_to(AltAz(obstime=t, location=location))

        return (altaz.alt, altaz.az)

# to_AA(120, 0)

In [5]:
import numpy as np
from astropy.coordinates import SkyCoord, EarthLocation, AltAz
from astropy.time import Time
import astropy.units as u

def to_AA(l_deg, b_deg, *, lat_obs=37.87, lon_obs=-122.27, height_m=0, obstime=None):
    """
    Convert Galactic (l,b) in degrees -> (Alt, Az) for an observer in Berkeley.

    Parameters
    ----------
    l_deg, b_deg : float
        Galactic longitude l and latitude b in degrees.
    lat_obs, lon_obs : float
        Observer latitude/longitude in degrees.
    height_m : float
        Observer height in meters.
    obstime : astropy.time.Time or None
        If None, uses current UTC time.

    Returns
    -------
    alt, az : astropy.coordinates.Angle
        Altitude and azimuth angles (use .deg for degrees).
    """

    # Galactic to ICRS (RA, dec)
    coord_gal = SkyCoord(l=l_deg * u.deg, b=b_deg * u.deg, frame="galactic")
    coord_icrs = coord_gal.icrs

    # Observer location
    location = EarthLocation(lat=lat_obs * u.deg, lon=lon_obs * u.deg, height=height_m * u.m)

    # Time
    if obstime is None:
        obstime = Time.now()

    # ICRS -> AltAz
    altaz = coord_icrs.transform_to(AltAz(obstime=obstime, location=location))

    return altaz.alt, altaz.az

alt, az = to_AA(120, 0)
print("Alt (deg):", alt.deg)
print("Az  (deg):", az.deg)

Alt (deg): 44.416566191805366
Az  (deg): 324.7329060207259


In [20]:
# Chat save us please 

import numpy as np
import ugradio

def to_rect(lon_deg, lat_deg):
    """
    lon, lat in DEGREES -> 3D unit vector (x,y,z).
    """
    lon = np.deg2rad(lon_deg)
    lat = np.deg2rad(lat_deg)

    x = np.zeros(3, dtype=float)
    x[0] = np.cos(lat) * np.cos(lon)
    x[1] = np.cos(lat) * np.sin(lon)
    x[2] = np.sin(lat)
    return x

def HA(l_deg, b_deg, elevation=0):
    """
    Galactic (l,b) in degrees -> local horizon-frame VECTOR (not yet alt/az angles).
    Returns a 3-vector in the final local frame.
    """

    # Galactic lon/lat -> Cartesian
    x_gal = to_rect(l_deg, b_deg)

    # Equatorial -> Galactic (J2000) rotation (your numbers, slightly more standard precision is OK too)
    EQ_TO_GAL = np.array([
        [-0.054876, -0.873437, -0.483835],
        [ 0.494109, -0.444830,  0.746982],
        [-0.867666, -0.198076,  0.455984],
    ], dtype=float)

    # Galactic -> Equatorial
    GAL_TO_EQ = np.linalg.inv(EQ_TO_GAL)
    x_eq = GAL_TO_EQ @ x_gal

    # --------
    # LST
    # --------
    lst = ugradio.timing.lst()  # often HOURS in ugradio
    # convert hours -> radians (robust heuristic)
    if lst > 2*np.pi:
        lst_rad = (lst / 24.0) * 2*np.pi
    else:
        lst_rad = lst

    # "RD_to_HD": rotate equatorial vector by +LST about z
    RD_to_HD = np.array([
        [ np.cos(lst_rad),  np.sin(lst_rad), 0.0],
        [-np.sin(lst_rad),  np.cos(lst_rad), 0.0],
        [ 0.0,              0.0,             1.0],
    ], dtype=float)

    x_hd = RD_to_HD @ x_eq

    # --------
    # site latitude
    # --------
    phi = np.deg2rad(37.8732)  # radians

    # "HD_to_AA": your matrix (keep it) but apply matrix @ vector
    HD_to_AA = np.array([
        [-np.sin(phi), 0.0,  np.cos(phi)],
        [ 0.0,        -1.0,  0.0],
        [ np.cos(phi), 0.0,  np.sin(phi)],
    ], dtype=float)

    x_local = HD_to_AA @ x_hd
    return x_local

v = HA(120, 0)
print("local vector:", v)

local vector: [0.63581025 0.44553977 0.63026994]


In [21]:
def vec_to_altaz(v):
    x, y, z = v
    alt = np.arcsin(np.clip(z, -1, 1))
    az  = np.arctan2(y, x) % (2*np.pi)
    return np.rad2deg(alt), np.rad2deg(az)

alt_deg, az_deg = vec_to_altaz(HA(120, 0))
print("alt (deg):", alt_deg, "az (deg):", az_deg)

alt (deg): 39.04694971485642 az (deg): 35.016039150510004


In [22]:
# And now to see if that is consistent with astropy again

alt, az = to_AA(120, 0)
print("Alt (deg):", alt.deg)
print("Az  (deg):", az.deg)

Alt (deg): 39.169726224389976
Az  (deg): 325.1527343451041


In [23]:
# Epic, code for pointing the telescope is written now. In a perfect world I would get the package to work but I am lazy

In [24]:
# START OF SECTION 8|